# AI Startup Predictor Fine-Tuning

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score
df = pd.read_csv('../datasets/startup_finetune_dataset.csv')
df.shape

In [ ]:
def get_value(k, t):
    m = re.search(rf'{k}\s*:\s*(.+)', str(t), re.I)
    return m.group(1).strip() if m else ''
def parse_prompt(t):
    name = get_value('Startup', t)
    cat = get_value('Category', t)
    desired = get_value('Desired fund', t)
    current = get_value('Current fund', t)
    status = get_value('Status', t)
    desc = get_value('Description', t)
    def to_num(x):
        try:
            return float(str(x).replace(',', '').strip())
        except:
            return 0.0
    return pd.Series({
        'name': name,
        'category': cat,
        'desired_fund': to_num(desired),
        'current_fund': to_num(current),
        'status': status,
        'description': desc,
        'funding_ratio': (to_num(current) / to_num(desired)) if to_num(desired) > 0 else 0.0
    })
def parse_completion(t):
    lines = [l.strip() for l in str(t).split('\n') if l.strip()]
    m = {k.strip().lower(): v.strip() for k,v in [tuple(s.split(':',1)) for s in lines if ':' in s]}
    s = m.get('attractiveness', '0')
    r = m.get('risk', '')
    a = m.get('reason', '')
    try:
        s = int(s)
    except:
        s = 0
    return pd.Series({'score': s, 'risk': r, 'reason': a})
Xp = df['prompt'].apply(parse_prompt)
Yc = df['completion'].apply(parse_completion)
data = pd.concat([Xp, Yc], axis=1)
data.head()

In [ ]:
X = data[['category','desired_fund','current_fund','status','description','funding_ratio']]
y_score = data['score']
y_risk = data['risk']
preprocess = ColumnTransformer([
    ('num', 'passthrough', ['desired_fund','current_fund','funding_ratio']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['category','status']),
    ('txt', TfidfVectorizer(max_features=3000), 'description')
])
reg = Pipeline([('prep', preprocess), ('model', Ridge())])
clf = Pipeline([('prep', preprocess), ('model', LogisticRegression(max_iter=1000))])
X_tr, X_te, ys_tr, ys_te = train_test_split(X, y_score, test_size=0.2, random_state=42)
X_tr2, X_te2, yr_tr, yr_te = train_test_split(X, y_risk, test_size=0.2, random_state=42)
reg.fit(X_tr, ys_tr)
clf.fit(X_tr2, yr_tr)
ys_pred = reg.predict(X_te)
yr_pred = clf.predict(X_te2)
mae = float(mean_absolute_error(ys_te, ys_pred))
rmse = float(mean_squared_error(ys_te, ys_pred) ** 0.5)
r2 = float(r2_score(ys_te, ys_pred))
acc = float(accuracy_score(yr_te, yr_pred))
f1 = float(f1_score(yr_te, yr_pred, average='macro'))
mae, rmse, r2, acc, f1

In [ ]:
def predict_analysis(model_reg, model_clf, name, category, desired_fund, current_fund, status, description):
    ratio = (current_fund / desired_fund) if desired_fund > 0 else 0.0
    row = pd.DataFrame([{
        'category': category,
        'desired_fund': desired_fund,
        'current_fund': current_fund,
        'status': status,
        'description': description,
        'funding_ratio': ratio
    }])
    s = float(model_reg.predict(row)[0])
    r = model_clf.predict(row)[0]
    adv = []
    if s < 70:
        adv.append('Clarify revenue model')
    if ratio < 0.3:
        adv.append('Reduce initial funding target')
    return {'attractiveness': int(round(s)), 'risk': r, 'advice': adv}
res = predict_analysis(reg, clf, 'EdTech platform', 'Education', 50000, 10000, 'active', 'Online learning for rural areas')
res

Done